## Step 1: Data Preparation & Preprocessing Logic
* **Identifier Feature Removal:** Dropped `RowNumber`, `CustomerId`, and `Surname` because non-informative features add noise and cause overfitting.
* **Missing Value Imputation:** Imputed missing values in `tenure` using the median value to retain records without skewing feature distributions.

In [6]:
import pandas as pd
import numpy as np

# 1. Load Dataset
df = pd.read_csv('/datasets/Churn.csv')

# 2. Drop identifiers (RowNumber, CustomerId, Surname carry zero predictive signal and cause overfitting)
df_clean = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# 3. Standardize column names to lowercase
df_clean.columns = [col.lower() for col in df_clean.columns]

# 4. Handle missing values in 'tenure' using median imputation
df_clean['tenure'] = df_clean['tenure'].fillna(df_clean['tenure'].median())

# Display dataset info to verify cleaning

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   creditscore      10000 non-null  int64  
 1   geography        10000 non-null  object 
 2   gender           10000 non-null  object 
 3   age              10000 non-null  int64  
 4   tenure           10000 non-null  float64
 5   balance          10000 non-null  float64
 6   numofproducts    10000 non-null  int64  
 7   hascrcard        10000 non-null  int64  
 8   isactivemember   10000 non-null  int64  
 9   estimatedsalary  10000 non-null  float64
 10  exited           10000 non-null  int64  
dtypes: float64(3), int64(6), object(2)
memory usage: 859.5+ KB


## Step 2: Class Imbalance Analysis & Data Splitting
* **Target Class Ratio:** The dataset exhibits a ~4:1 class imbalance (~79.6% retained vs. ~20.4% churned).
* **Data Splitting:** Stratified split into 60% Train, 20% Validation, and 20% Test sets to preserve the target ratio across all sets.

In [7]:
from sklearn.model_selection import train_test_split

# 1. Check class distribution (Target: exited)
class_counts = df_clean['exited'].value_counts(normalize=True)
print("Class Distribution:\n", class_counts)

# 2. Separate features (X) and target (y)
X = df_clean.drop(columns=['exited'])
y = df_clean['exited']

# 3. Split data into Train (60%), Validation (20%), and Test (20%) sets
# First split: 80% train+val, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second split: 60% train, 20% val (0.25 of 80% = 20% of total dataset)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# Verify set sizes
print(f"\nTraining set size:   {len(X_train)} rows")
print(f"Validation set size: {len(X_val)} rows")
print(f"Test set size:       {len(X_test)} rows")

Class Distribution:
 0    0.7963
1    0.2037
Name: exited, dtype: float64

Training set size:   6000 rows
Validation set size: 2000 rows
Test set size:       2000 rows


## Step 3: Feature Encoding & Baseline Model Findings
* **Categorical Encoding:** Encoded categorical features using `OneHotEncoder(drop='first')` to prevent multicollinearity without adding unseen-category errors.
* **Baseline Result:** Training without addressing class imbalance yields an F1 score of only 0.4871, showing that standard algorithms favor the majority class without rebalancing.

In [8]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score

# 1. Categorical & Numerical Feature Pipeline
categorical_cols = ['geography', 'gender']
numerical_cols = [
    'creditscore', 'age', 'tenure', 'balance', 
    'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary'
]

# Using sparse=False for older scikit-learn compatibility
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', sparse=False), categorical_cols)
])

# 2. Fit preprocessor ONLY on training set, then transform validation set
X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep = preprocessor.transform(X_val)

# 3. Train Baseline Decision Tree (Ignoring Imbalance)
baseline_model = DecisionTreeClassifier(random_state=42)
baseline_model.fit(X_train_prep, y_train)

# 4. Evaluate Baseline on Validation Set
y_val_pred_base = baseline_model.predict(X_val_prep)
y_val_proba_base = baseline_model.predict_proba(X_val_prep)[:, 1]

print("--- BASELINE RESULTS (UNBALANCED) ---")
print(f"Validation F1 Score: {f1_score(y_val, y_val_pred_base):.4f}")
print(f"Validation ROC-AUC:  {roc_auc_score(y_val, y_val_proba_base):.4f}")

--- BASELINE RESULTS (UNBALANCED) ---
Validation F1 Score: 0.4871
Validation ROC-AUC:  0.6780


## Step 4: Class Imbalance Mitigation & Model Tuning
* **Techniques Evaluated:** Applied Class Weighting (`class_weight='balanced'`), Upsampling, and Downsampling across standard decision tree and Random Forest models.
* **Top Configuration:** A Class-Weighted Random Forest (`max_depth=10`, `n_estimators=100`) achieved the highest validation F1 score (~0.6386).

In [9]:
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier

# --- 1. Define Resampling Functions ---
def upsample(X, y, repeat=4):
    X_df = pd.DataFrame(X)
    X_df['target'] = y.values
    df_majority = X_df[X_df['target'] == 0]
    df_minority = X_df[X_df['target'] == 1]
    
    df_minority_upsampled = resample(
        df_minority, replace=True, n_samples=len(df_minority) * repeat, random_state=42
    )
    df_upsampled = pd.concat([df_majority, df_minority_upsampled])
    return df_upsampled.drop(columns=['target']).values, df_upsampled['target'].values

def downsample(X, y, fraction=0.25):
    X_df = pd.DataFrame(X)
    X_df['target'] = y.values
    df_majority = X_df[X_df['target'] == 0]
    df_minority = X_df[X_df['target'] == 1]
    
    df_majority_downsampled = df_majority.sample(frac=fraction, random_state=42)
    df_downsampled = pd.concat([df_majority_downsampled, df_minority])
    return df_downsampled.drop(columns=['target']).values, df_downsampled['target'].values

# Create resampled datasets from training data
X_train_up, y_train_up = upsample(X_train_prep, y_train)
X_train_down, y_train_down = downsample(X_train_prep, y_train)

# --- 2. Grid Search Tuning Across Imbalance Approaches ---
best_f1 = 0
best_model = None
best_config = ""
best_params = {}

# Technique A: Class-Weighted Random Forest
for n_est in [50, 100, 150]:
    for depth in [6, 8, 10, 12]:
        rf = RandomForestClassifier(
            n_estimators=n_est, max_depth=depth, class_weight='balanced', random_state=42
        )
        rf.fit(X_train_prep, y_train)
        preds = rf.predict(X_val_prep)
        score = f1_score(y_val, preds)
        if score > best_f1:
            best_f1 = score
            best_model = rf
            best_config = "Class-Weighted"
            best_params = {'n_estimators': n_est, 'max_depth': depth}

# Technique B: Upsampled Random Forest
for n_est in [50, 100, 150]:
    for depth in [6, 8, 10, 12]:
        rf_up = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=42)
        rf_up.fit(X_train_up, y_train_up)
        preds = rf_up.predict(X_val_prep)
        score = f1_score(y_val, preds)
        if score > best_f1:
            best_f1 = score
            best_model = rf_up
            best_config = "Upsampled"
            best_params = {'n_estimators': n_est, 'max_depth': depth}

# Technique C: Downsampled Random Forest
for n_est in [50, 100, 150]:
    for depth in [6, 8, 10, 12]:
        rf_down = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=42)
        rf_down.fit(X_train_down, y_train_down)
        preds = rf_down.predict(X_val_prep)
        score = f1_score(y_val, preds)
        if score > best_f1:
            best_f1 = score
            best_model = rf_down
            best_config = "Downsampled"
            best_params = {'n_estimators': n_est, 'max_depth': depth}

print("--- TUNING RESULTS ---")
print(f"Best Strategy: {best_config}")
print(f"Best Parameters: {best_params}")
print(f"Best Validation F1 Score: {best_f1:.4f}")

--- TUNING RESULTS ---
Best Strategy: Class-Weighted
Best Parameters: {'n_estimators': 100, 'max_depth': 10}
Best Validation F1 Score: 0.6385


## Step 5: Final Model Testing & Evaluation Metrics
* **F1 Score Result:** Reached an F1 score of **0.6259** on the unseen test set, passing the required 0.59 threshold.
* **AUC-ROC Comparison:** Achieved an AUC-ROC score of **0.8639**. The high AUC-ROC confirms the model reliably separates churned vs. retained customers across decision thresholds.

In [10]:
# 1. Refit preprocessor on combined Train + Validation data to leverage full training set
X_train_val_prep = preprocessor.fit_transform(X_train_val)
X_test_prep = preprocessor.transform(X_test)

# 2. Fit the best model configuration on the combined dataset
final_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42
)
final_model.fit(X_train_val_prep, y_train_val)

# 3. Predict on unseen Test set
y_test_pred = final_model.predict(X_test_prep)
y_test_proba = final_model.predict_proba(X_test_prep)[:, 1]

test_f1 = f1_score(y_test, y_test_pred)
test_auc_roc = roc_auc_score(y_test, y_test_proba)

print("=== FINAL TEST RESULTS ===")
print(f"Test F1 Score:  {test_f1:.4f} (Requirement: >= 0.59)")
print(f"Test AUC-ROC:   {test_auc_roc:.4f}")

=== FINAL TEST RESULTS ===
Test F1 Score:  0.6259 (Requirement: >= 0.59)
Test AUC-ROC:   0.8639
